# ممیزی اولیهٔ Qwen3.5-2B برای پروژهٔ MoE-0.4B

این Notebook فقط پیکربندی عمومی مدل را می‌خواند؛ در این مرحله وزن‌های ۴.۵ گیگابایتی دانلود نمی‌شوند و هیچ آموزش یا محاسبهٔ پولی انجام نمی‌شود.

**هدف:** تأیید معماری واقعی، محاسبهٔ برآوردهای اولیه و ذخیرهٔ گزارش قابل‌بازتولید.

In [ ]:
import json, urllib.request, datetime
MODEL_ID = 'Qwen/Qwen3.5-2B'
CONFIG_URL = f'https://huggingface.co/{MODEL_ID}/resolve/main/config.json'
req = urllib.request.Request(CONFIG_URL, headers={'User-Agent': 'Qwen-MoE-0.4B-audit/0.1'})
with urllib.request.urlopen(req, timeout=30) as response:
    config = json.load(response)
print('Model:', MODEL_ID)
print('Architecture:', config.get('architectures'))
print('Model type:', config.get('model_type'))
print('Config fetched successfully; model weights were not downloaded.')

In [ ]:
text = config.get('text_config', config)
hidden = int(text['hidden_size'])
vocab = int(text['vocab_size'])
intermediate = int(text['intermediate_size'])
layers = int(text['num_hidden_layers'])
layer_types = text.get('layer_types', [])
print('Hidden size:', hidden)
print('Vocabulary:', vocab)
print('Layers:', layers)
print('Intermediate size:', intermediate)
print('Layer type counts:', {name: layer_types.count(name) for name in sorted(set(layer_types))})
print('Tied embeddings:', text.get('tie_word_embeddings', config.get('tie_word_embeddings')))
print('Embedding matrix estimate:', f'{hidden*vocab:,}', f'({hidden*vocab/1e6:.1f}M parameters)')

## برآورد فرضیهٔ MoE

اعداد زیر فقط ماتریس‌های gate/up/down با فرض SwiGLU را می‌شمارند؛ Attention، Router، نرمال‌سازی، bias و هزینهٔ logits را شامل نمی‌شوند. این اعداد را نباید «پارامتر فعال کل مدل» نامید.

In [ ]:
NUM_EXPERTS = 8
TOP_K = 2
EXPERT_INTERMEDIATE = 768
params_per_expert = 3 * hidden * EXPERT_INTERMEDIATE
total_expert_ffn = params_per_expert * NUM_EXPERTS * layers
active_expert_ffn = params_per_expert * TOP_K * layers
dense_ffn_estimate = 3 * hidden * intermediate * layers
report = {
    'model_id': MODEL_ID,
    'audited_at_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'architecture': config.get('architectures'),
    'text_config': {'hidden_size': hidden, 'vocab_size': vocab, 'num_hidden_layers': layers,
                    'intermediate_size': intermediate, 'layer_types': layer_types},
    'estimates': {'tied_embedding_parameters_if_tied': hidden*vocab,
                  'dense_swiglu_ffn_all_layers': dense_ffn_estimate,
                  'experts': NUM_EXPERTS, 'top_k': TOP_K,
                  'expert_intermediate_size': EXPERT_INTERMEDIATE,
                  'all_expert_ffn_parameters': total_expert_ffn,
                  'active_expert_ffn_parameters_per_token': active_expert_ffn},
    'scope': 'Config-only; no weights downloaded; no training performed.'
}
print(json.dumps(report, ensure_ascii=False, indent=2))
with open('qwen35_moe_audit_report.json', 'w', encoding='utf-8') as f:
    json.dump(report, f, ensure_ascii=False, indent=2)
print('Saved qwen35_moe_audit_report.json')

## تصمیم پس از ممیزی

۱. گزارش و نسخهٔ پیکربندی را ثبت کنید.
۲. سپس در یک سلول جداگانه، سازگاری نسخهٔ Transformers و tokenizer را بررسی می‌کنیم.
۳. بارگذاری کامل وزن‌ها فقط پس از بررسی RAM/VRAM محیط انجام می‌شود.
۴. اولین نمونهٔ MoE فقط روی یک لایه و با آزمون forward/backward شروع خواهد شد.

**محدودیت مهم:** embedding مشترک برآوردی به‌تنهایی حدود ۵۰۸.۶ میلیون پارامتر است؛ بنابراین سقف ۰.۴B برای کل مدل با حفظ ابعاد فعلی شدنی نیست. باید مشخص کنیم منظورمان پارامتر فعال FFN است یا باید دانش‌آموز کوچک‌تری طراحی کنیم.